# CAPTIONING D'IMAGES - WORKSHOP LIVRABLE 3

In [3]:
import tensorflow as tf

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import collections
import random
import re
import numpy as np
import os
import time
import json
from glob import glob
from PIL import Image
import pickle
from tqdm import tqdm

2025-04-20 22:11:14.643072: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745179874.743643    2546 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745179874.775647    2546 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745179875.030034    2546 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745179875.030071    2546 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745179875.030074    2546 computation_placer.cc:177] computation placer alr

## Introduction
"""
Ce notebook présente la solution de génération automatique de légendes pour des photographies
à l'aide d'un réseau de neurones combinant CNN (Convolutional Neural Network) et RNN (Recurrent Neural Network).
Le dataset utilisé est MS COCO.
"""

## 1. Chargement des données
##### - Charger les images et les légendes associées depuis MS COCO
##### - Afficher quelques exemples


In [ ]:
# Chemin du fichier d'annotations
annotation_folder = "../../annotations/"
annotation_file = annotation_folder +"captions_train2014.json"

# Chemin du dossier contenant les images à annoter
image_folder = '../../train2014/'

print("annotation file ", annotation_file)

# Lecture du fichier d'annotation
with open(annotation_file, 'r') as f:
    annotations = json.load(f)

with open(annotation_folder+"instances_val2014.json", 'r') as f:
    instances = json.load(f)

with open(annotation_folder+"/captions_val2014.json", 'r') as f:
    annotations2 = json.load(f)

# Grouper toutes les annotations ayant le même identifiant.
image_path_to_caption = collections.defaultdict(list)
for val in annotations['annotations']:
    # marquer le début et la fin de chaque annotation
    caption = '<start> ' + val['caption'] + ' <end>'
    # L'identifiant d'une image fait partie de son chemin d'accès
    image_path = image_folder + 'COCO_train2014_' + '%012d.jpg' % (val['image_id'])
    # Rajout du caption associé à image_path
    image_path_to_caption[image_path].append(caption)
    
# Prendre les premières images seulement
image_paths = list(image_path_to_caption.keys())
random.shuffle(image_paths)
train_image_paths = image_paths[:2000]
print(len(train_image_paths))

# Liste de toutes les annotations
train_captions = []
# Liste de tous les noms de fichiers des images dupliquées (en nombre d'annotations par image)
img_name_vector = []

for image_path in train_image_paths:
    caption_list = image_path_to_caption[image_path]
    # Rajout de caption_list dans train_captions
    train_captions.extend(caption_list)
    # Rajout de image_path dupliquée len(caption_list) fois
    img_name_vector.extend([image_path] * len(caption_list))


annotation file  ../../annotations/captions_train2014.json
2000


In [16]:
import os
print(os.path.exists(image_folder + "COCO_train2014_000000581795.jpg"))

True


In [18]:
print(img_name_vector[0])

../../train2014/COCO_train2014_000000501898.jpg


In [20]:
print(train_captions[0])

<start> A woman standing on a tennis court holding a racquet. <end>


In [24]:
for i in range(5):
    print(img_name_vector[i])

../../train2014/COCO_train2014_000000501898.jpg
../../train2014/COCO_train2014_000000501898.jpg
../../train2014/COCO_train2014_000000501898.jpg
../../train2014/COCO_train2014_000000501898.jpg
../../train2014/COCO_train2014_000000501898.jpg


In [23]:
print(len(train_captions), len(img_name_vector))
print(train_captions[0])
Image.open(img_name_vector[3])

10007 10007
<start> A woman standing on a tennis court holding a racquet. <end>


FileNotFoundError: [Errno 2] No such file or directory: '../../train2014/COCO_train2014_000000501898.jpg'

## 2. Prétraitements 
### 2.1 Prétraitement des images
##### - Redimensionnement, normalisation
##### - Extraction des features via un CNN pré-entraîné (e.g., InceptionV3)

### 2.2 Prétraitement du texte
##### - Nettoyage des légendes (ponctuation, minuscules, etc.)
##### - Tokenisation et indexation
##### - Séparation input/target et padding


## 3. Construction du modèle
### 3.1 Partie CNN : encodeur d'image
##### - Utiliser un modèle pré-entraîné pour extraire les features

### 3.2 Partie RNN : générateur de légende
##### - Embedding -> LSTM -> Dense softmax
##### - Prise en compte du vecteur image au début ou par concaténation

## 4. Entraînement
##### - Compilation du modèle (loss, optimizer)
##### - Entraînement avec monitoring de la loss et accuracy
##### - Sauvegarde du modèle

## 5. Évaluation des performances
##### - Affichage des courbes de loss et accuracy
##### - Test sur quelques images : affichage de la vraie légende vs. celle générée


## 6. Améliorations possibles
##### - Ajouter un mécanisme d'attention
##### - Utiliser Beam Search pour la prédiction
##### - Entraîner sur plus de données ou avec augmentation